In [1]:
from keras.layers import Activation, Dense, Dropout, SpatialDropout1D
from keras.layers import Embedding
from keras.layers import LSTM
from keras.models import Sequential
from keras.preprocessing import sequence
from sklearn.model_selection import train_test_split
import collections
import matplotlib.pyplot as plt
import nltk
import numpy as np
import os

In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od

In [ ]:
!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {"username":"csweetlinhemalatha","key":"4e294466fab9af8acc46f174727d628c"}

import json

with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d https://www.kaggle.com/c/si650winter11/data

Invalid dataset specification https://www.kaggle.com/c/si650winter11/data?select=training.txt


In [ ]:
import kaggle

OSError: Could not find kaggle.json. Make sure it's located in /root/.config/kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/

In [2]:
from google.colab import files

files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"csweetlinhemalatha","key":"4e294466fab9af8acc46f174727d628c"}'}

In [5]:
!mkdir ~/.kaggle

In [6]:
! cp kaggle.json ~/.kaggle/

In [7]:
!kaggle competitions download -c si650winter11

  0% 0.00/506k [00:00<?, ?B/s]
100% 506k/506k [00:00<00:00, 112MB/s]


In [ ]:
!kaggle datasets list

ref                                                           title                                              size  lastUpdated          downloadCount  voteCount  usabilityRating  
------------------------------------------------------------  ------------------------------------------------  -----  -------------------  -------------  ---------  ---------------  
anandshaw2001/netflix-movies-and-tv-shows                     Netflix Movies and TV Shows                         1MB  2025-01-03 10:33:01           6917        191  1.0              
ashaychoudhary/anxiety-attack-factors-symptoms-and-severity   Anxiety Attack : Factors, Symptoms, and Severity  244KB  2025-01-19 11:56:21            766         22  1.0              
ayushcx/apple-sales-dataset-2024                              Apple Sales Dataset (2024)                         13KB  2025-01-20 10:08:33            878         25  0.88235295       
ankushpanday1/tuberculosis-tb-predictiontop-75-countries      Tuberculosis (TB) 

replace /root/Dataset/testdata.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [8]:
!unzip -q /content/si650winter11.zip -d Dataset

In [13]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [36]:
#preprocssing to know how many unique words there are in the corpus and how many words are there in each sentence
maxlen = 0
word_freqs = collections.Counter()
num_recs = 0
ftrain = open('/content/Dataset/training.txt', 'r')
for line in ftrain:
  label, sentence = line.strip().split("\t")
  #words = nltk.word_tokenize(sentence.decode("ascii", "ignore").lower())
  words = nltk.word_tokenize(sentence.lower())
  if len(words) > maxlen:
    maxlen = len(words)
  for word in words:
      word_freqs[word] += 1
  num_recs += 1
ftrain.close()

In [37]:
word_freqs

Counter({'the': 3221,
         'da': 1998,
         'vinci': 2001,
         'code': 1994,
         'book': 135,
         'is': 1521,
         'just': 287,
         'awesome': 1126,
         '.': 3364,
         'this': 213,
         'was': 1179,
         'first': 104,
         'clive': 1,
         'cussler': 1,
         'i': 4707,
         "'ve": 11,
         'ever': 96,
         'read': 113,
         ',': 4194,
         'but': 295,
         'even': 27,
         'books': 28,
         'like': 974,
         'relic': 1,
         'and': 2150,
         'were': 96,
         'more': 103,
         'plausible': 1,
         'than': 14,
         'liked': 101,
         'a': 1305,
         'lot': 22,
         'it': 899,
         'ultimatly': 1,
         'did': 33,
         "n't": 224,
         'seem': 1,
         'to': 808,
         'hold': 2,
         "'s": 629,
         'own': 4,
         'that': 718,
         'not': 198,
         'an': 225,
         'exaggeration': 1,
         ')': 44,
         '

In [38]:
len(word_freqs)

2268

In [39]:
maxlen

42

In [40]:
num_recs

7086

In [18]:
#BASED ON PRECEEDING ESTIMATE WE SET THE VALUES
MAX_FEATURES = 2000 #remaining words are out of vocabulary words OOV
MAX_SENTENCE_LENGTH = 40  #sets a fixed sequence length and zero pad shorter or longer sequence

In [41]:
#creating pair of lookup tables - word2index and index2word
vocab_size = min(MAX_FEATURES, len(word_freqs)) + 2   #vocabulary of size 2002
word2index = {x[0]: i+2 for i, x in enumerate(word_freqs.most_common(MAX_FEATURES))}
word2index["PAD"] = 0
word2index["UNK"] = 1
index2word = {v:k for k, v in word2index.items()}

In [56]:
len(word2index)

2002

In [34]:
word2index['talks']

1426

In [35]:
index2word[1426]

'talks'

In [50]:
a=np.empty((5, ),dtype=list)
a
len(a)

5

In [58]:
#convert our input sentences to word index sequences, pad them to the MAX_SENTENCE_LENGTH words
X = np.empty((num_recs, ), dtype=list)
y = np.zeros((num_recs, ))
i = 0
ftrain = open('/content/Dataset/training.txt', 'r')
for line in ftrain:
  label, sentence = line.strip().split("\t")
  #words = nltk.word_tokenize(sentence.decode("ascii", "ignore").lower())
  words = nltk.word_tokenize(sentence.lower())
  seqs = []
  for word in words:
    if word in word2index:
      seqs.append(word2index[word])
    else:
      seqs.append(word2index["UNK"])
    X[i] = seqs
    y[i] = int(label)
    i += 1
ftrain.close()
X = sequence.pad_sequences(X, maxlen=MAX_SENTENCE_LENGTH)



IndexError: index 7086 is out of bounds for axis 0 with size 7086

In [55]:
y.shape

(7086,)

In [ ]:
#split the training set into a 80-20 training test split
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

In [57]:
#initializing variables
EMBEDDING_SIZE = 128
HIDDEN_LAYER_SIZE = 64
BATCH_SIZE = 32
NUM_EPOCHS = 10

In [ ]:
#model building
model = Sequential()
model.add(Embedding(vocab_size, EMBEDDING_SIZE,
input_length=MAX_SENTENCE_LENGTH))
model.add(SpatialDropout1D(Dropout(0.2)))
model.add(LSTM(HIDDEN_LAYER_SIZE, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1))
model.add(Activation("sigmoid"))

In [ ]:
model.compile(loss="binary_crossentropy", optimizer="adam",
metrics=["accuracy"])

In [ ]:
#train the network
history = model.fit(Xtrain, ytrain, batch_size=BATCH_SIZE, epochs=NUM_EPOCHS,
validation_data=(Xtest, ytest))

In [ ]:
#plot accuracy and loss
plt.subplot(211)
plt.title("Accuracy")
plt.plot(history.history["acc"], color="g", label="Train")
plt.plot(history.history["val_acc"], color="b", label="Validation")
plt.legend(loc="best")
plt.subplot(212)
plt.title("Loss")
plt.plot(history.history["loss"], color="g", label="Train")
plt.plot(history.history["val_loss"], color="b", label="Validation")
plt.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
#evaluate
score, acc = model.evaluate(Xtest, ytest, batch_size=BATCH_SIZE)
print("Test score: %.3f, accuracy: %.3f" % (score, acc))


In [ ]:
#predict
for i in range(5):
  idx = np.random.randint(len(Xtest))
  xtest = Xtest[idx].reshape(1,40)
  ylabel = ytest[idx]
  ypred = model.predict(xtest)[0][0]
  sent = " ".join([index2word[x] for x in xtest[0].tolist() if x != 0])
  print("%.0ft%dt%s" % (ypred, ylabel, sent))